# ~1B baseline on a T4

Runs **Qwen3.5-0.8B**, untuned, through the same harness and settings as the teacher baseline `qwen3.8-27b-fp8-v5`, on the `dev` suite.

Qwen3.8 has no small model, so Qwen3.5-0.8B is the nearest ~1B Qwen. It is the model the ~1B student will be trained from.

| | teacher | this run |
|---|---|---|
| model | Qwen3.8-27B | Qwen3.5-0.8B |
| weights | FP8 | fp16 (a T4 has no FP8 or bf16) |
| server | vLLM | vLLM |
| thinking | none | none |
| temperature | 0 | 0 |
| tool rounds / tool tokens / answer tokens | 10 / 800 / 1500 | 10 / 800 / 1500 (the defaults) |
| cases in flight | 16 (`--max-num-seqs 32`) | 16 (`--max-num-seqs 32`) |
| suite | `dev`, 280 cases | `dev`, 280 cases |

Use **Runtime → Change runtime type → T4 GPU**. The notebook clones `REPO_URL` at `BRANCH`, [kurexi/cost](https://github.com/kurexi/cost) by default; point them at a fork to run your own changes (for a private one, add a Colab secret named `GITHUB_TOKEN` with read access to it). `AGENT` is the agent's folder inside the clone: empty for kurexi/cost, where the agent is the whole repository. `RECORDS` is the folder, inside the clone, that holds `runs/` and `RESULTS.md`; `eval-records` by default, the repository's own runs; leave it empty to keep them in the agent's `eval/` folder. The eval scripts find it through `SPENDING_EVAL_RECORDS`.

In [ ]:
!nvidia-smi

In [ ]:
import os
import subprocess
from google.colab import userdata

REPO_URL = "https://github.com/kurexi/cost"
BRANCH = "main"
RECORDS = "eval-records"
AGENT = ""
CLONE = "/content/repo"
EVAL = os.path.join(CLONE, AGENT, "eval")
RECORDS_DIR = f"{CLONE}/{RECORDS}" if RECORDS else EVAL
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = ""
url = REPO_URL.replace("https://", f"https://{token}@", 1) if token else REPO_URL
os.chdir("/content")
subprocess.run(["rm", "-rf", CLONE], check=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, CLONE], check=True)
os.environ["SPENDING_EVAL_RECORDS"] = RECORDS_DIR
os.makedirs(f"{RECORDS_DIR}/runs", exist_ok=True)

In [ ]:
!pip install -q -U vllm

## Start the server

The model is served as `qwen`, like the teacher, with tool calling on (`qwen3_coder` is Qwen3.5's tool-call format). It runs in the background and logs to `/content/vllm.log`. The next cell waits until it answers (the first start downloads the weights and compiles, a few minutes).

The cell first stops any vLLM left over from an earlier attempt. A crashed engine process (`VLLM::EngineCore`) keeps its GPU memory, and the next server then fails with `No available memory for the cache blocks`. Memory used should read near zero before the server starts; if it doesn't, use **Runtime → Restart session**.

In [ ]:
!pkill -if "[v]llm"; sleep 5; nvidia-smi --query-gpu=memory.used,memory.total --format=csv
!nohup vllm serve Qwen/Qwen3.5-0.8B --served-model-name qwen --dtype half --max-model-len 16384 --max-num-seqs 32 --gpu-memory-utilization 0.85 --enable-auto-tool-choice --tool-call-parser qwen3_coder --reasoning-parser qwen3 --port 8090 > /content/vllm.log 2>&1 &

In [ ]:
import time
import urllib.request

deadline = time.time() + 1200
while True:
    try:
        urllib.request.urlopen("http://127.0.0.1:8090/v1/models", timeout=5)
        print("server is up")
        break
    except Exception:
        if time.time() > deadline:
            raise SystemExit(open("/content/vllm.log").read()[-4000:])
        time.sleep(10)

## Check the harness and the answer format

`oracle` must grade 1.000 and `silent` 0. `probe_formats.py` must accept the **tool round** (each session starts with one; without the server's tool-call parser every case fails with a 400) and the **full bounded schema**: a run that falls back to a looser format is not a fair baseline against the teacher, which answered every case on `json_schema`.

In [ ]:
%cd {EVAL}
!python3 run.py --suite dev --name check-oracle --backend oracle --restart | grep "^record_core"
!python3 run.py --suite dev --name check-silent --backend silent --restart | grep "^record_core"
!python3 probe_formats.py --base-url http://127.0.0.1:8090/v1 --model qwen

## The run

If Colab disconnects, restart the server cells and run this cell again: the run resumes from the last finished case.

In [ ]:
%cd {EVAL}
!python3 run.py --suite dev --name qwen3.5-0.8b-fp16 --base-url http://127.0.0.1:8090/v1 --model qwen --reasoning-effort none --workers 16

## Against the teacher

The first run is the baseline for the comparison; `board.py --write` adds the new row to `RESULTS.md`.

In [ ]:
%cd {EVAL}
!python3 compare.py qwen3.8-27b-fp8-v5 qwen3.5-0.8b-fp16
!python3 board.py --write

## Take the results home

Download the zip, then unpack it into your records folder and commit it: `runs/qwen3.5-0.8b-fp16/` and the updated `RESULTS.md`. Add a line for the run under *Notes per run*.

In [ ]:
%cd {RECORDS_DIR}
!zip -qr /content/qwen3.5-0.8b-fp16.zip runs/qwen3.5-0.8b-fp16 RESULTS.md
from google.colab import files
files.download("/content/qwen3.5-0.8b-fp16.zip")